# 00 — PostgreSQL Telemetry Time Buckets

This playground mirrors the MySQL prep style, but uses PostgreSQL for observability labs.


## Cell 2 — Install/import dependencies


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)


## Cell 3 — Connection settings


In [ ]:
DB_HOST = 'host.docker.internal'  # Jupyter container -> host Docker service
DB_PORT = 5432
DB_NAME = 'observability'
DB_USER = 'obs_user'
DB_PASS = 'obs_pass'

CONN_STR = f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(CONN_STR, pool_pre_ping=True)
CONN_STR


## Cell 4 — Smoke test connection


In [ ]:
with engine.connect() as conn:
    row = conn.execute(text('SELECT now() AS now_utc, current_database() AS db')).mappings().one()

row


## Cell 5 — Helper function to run SQL


In [ ]:
def run_sql(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(text(sql), engine)


## Cell 6 — Helper function to inspect one table safely


In [ ]:
def inspect_table_safe(schema_name: str, table_name: str) -> None:
    q = f'''
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = '{schema_name}'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position
    '''
    display(run_sql(q))

inspect_table_safe('lab', 'telemetry_cpu_raw')


## Cell 7 — First time-bucket query (hourly)


In [ ]:
sql = '''
SELECT
  date_trunc('hour', sampled_at) AS hour_bucket,
  host,
  ROUND(AVG(cpu_pct), 2) AS avg_cpu,
  MAX(cpu_pct) AS peak_cpu
FROM lab.telemetry_cpu_raw
WHERE env = 'prod'
GROUP BY 1, 2
ORDER BY hour_bucket DESC, host
LIMIT 25;
'''
run_sql(sql)


## Cell 8 — Window example (rolling 6-hour average)


In [ ]:
sql = '''
WITH hourly AS (
  SELECT date_trunc('hour', sampled_at) AS hour_bucket, host, AVG(cpu_pct) AS avg_cpu
  FROM lab.telemetry_cpu_raw
  GROUP BY 1,2
)
SELECT
  hour_bucket,
  host,
  ROUND(avg_cpu::numeric, 2) AS avg_cpu,
  ROUND(AVG(avg_cpu) OVER (
    PARTITION BY host
    ORDER BY hour_bucket
    ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
  )::numeric, 2) AS rolling_6h_avg
FROM hourly
ORDER BY host, hour_bucket
LIMIT 50;
'''
run_sql(sql)
